# Level 1 — Understand dataset structure

Before making charts, I want to know what one row means, how large the file is, and which columns look suspicious.

In [1]:
from pathlib import Path
import pandas as pd

# Works if I open Jupyter from the repo root or from notebooks/
data_path = Path("data/sales_raw.csv")
if not data_path.exists():
    data_path = Path("../data/sales_raw.csv")

df = pd.read_csv(data_path)
df.head(3)

,line_id,order_id,order_date,customer_id,city,category,product,quantity,unit_price_pkr,discount_pct,revenue_pkr,payment_method,order_status,channel,delivery_days
0,LN-00001,ORD-2025-25001,2025-05-13,C-1023,Lahore,Office,Whiteboard,4,6000,10%,21600.0,Card,Delivered,Store,10.0
1,LN-00002,ORD-2025-25001,2025-08-19,C-1134,Islamabad,Home,Wall Clock,5,3100,15%,13175.0,Cash,Delivered,Store,8.0
2,LN-00003,ORD-2025-25002,2025-09-10,C-1052,Rawalpindi,NaN,Desk Lamp,3,2592,0%,7776.0,Bank Transfer,Delivered,Online,6.0


In [2]:
print("shape:", df.shape)
print("size:", df.size)
print("ndim:", df.ndim)
print("index:", df.index[:5])

shape: (565, 15)
size: 8475
ndim: 2
index: RangeIndex(start=0, stop=5, step=1)


In [3]:
df.head(5)

,line_id,order_id,order_date,customer_id,city,category,product,quantity,unit_price_pkr,discount_pct,revenue_pkr,payment_method,order_status,channel,delivery_days
0,LN-00001,ORD-2025-25001,2025-05-13,C-1023,Lahore,Office,Whiteboard,4,6000,10%,21600.0,Card,Delivered,Store,10.0
1,LN-00002,ORD-2025-25001,2025-08-19,C-1134,Islamabad,Home,Wall Clock,5,3100,15%,13175.0,Cash,Delivered,Store,8.0
2,LN-00003,ORD-2025-25002,2025-09-10,C-1052,Rawalpindi,NaN,Desk Lamp,3,2592,0%,7776.0,Bank Transfer,Delivered,Online,6.0
3,LN-00004,ORD-2025-25002,2025-06-24,C-1108,Islamabad,Office,Whiteboard,2,6360,5%,12084.0,Cash,Delivered,Store,1.0
4,LN-00005,ORD-2025-25003,2025-07-21,C-1083,Lahore,Office,Pen Set,1,689,10%,620.1,Card,Delivered,WhatsApp,11.0


In [4]:
# Check a different part of the data too
df.tail(3)

,line_id,order_id,order_date,customer_id,city,category,product,quantity,unit_price_pkr,discount_pct,revenue_pkr,payment_method,order_status,channel,delivery_days
562,LN-00302,ORD-2025-25151,2025-02-08,C-1102,Rawalpindi,Accessories,Charging Cable,4,950,0%,3800.0,Wallet,Delivered,Online,11.0
563,LN-00089,ORD-2025-25045,21/02/2025,C-1053,Faisalabad,Electronics,USB-C Hub,2,4032,0%,8564.0,Bank Transfer,Delivered,Store,3.0
564,LN-00244,ORD-2025-25122,2025-05-21,C-1139,Lahore,Accessories,Power Bank,2,7392,0%,14784.0,Cash,Pending,Store,3.0


In [5]:
df.sample(5, random_state=7)

,line_id,order_id,order_date,customer_id,city,category,product,quantity,unit_price_pkr,discount_pct,revenue_pkr,payment_method,order_status,channel,delivery_days
258,LN-00259,ORD-2025-25130,09/04/2025,C-1075,Lahore,Electronics,Keyboard,2,3360,5%,6384.0,Card,Delivered,Online,11.0
539,LN-00540,ORD-2025-25270,11/09/2025,C-1166,Lahore,Accessories,Power Bank,4,7700,10%,27720.0,Cash,Pending,Online,NaN
97,LN-00098,ORD-2025-25049,2025-05-24,C-1032,Lahore,Electronics,Webcam,2,6900,0%,13800.0,Cash,Delivered,WhatsApp,8.0
435,LN-00436,ORD-2025-25218,2025-04-17,C-1005,Islamabad,Electronics,Wireless Mouse,3,1908,0%,5724.0,Bank Transfer,Delivered,Online,5.0
410,LN-00411,ORD-2025-25206,2025-04-12,C-1043,Karachi,Home,Water Bottle,2,950,20%,1520.0,Cash,Delivered,Store,3.0


### Datatypes and null overview

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 565 entries, 0 to 564
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   line_id         565 non-null    object 
 1   order_id        565 non-null    object 
 2   order_date      565 non-null    object 
 3   customer_id     556 non-null    object 
 4   city            552 non-null    object 
 5   category        547 non-null    object 
 6   product         565 non-null    object 
 7   quantity        565 non-null    object 
 8   unit_price_pkr  565 non-null    int64  
 9   discount_pct    532 non-null    object 
 10  revenue_pkr     550 non-null    float64
 11  payment_method  541 non-null    object 
 12  order_status    565 non-null    object 
 13  channel         565 non-null    object 
 14  delivery_days   448 non-null    float64
dtypes: float64(2), int64(1), object(12)
memory usage: 66.3+ KB


In [7]:
df.dtypes.to_frame("dtype")

,dtype
line_id,object
order_id,object
order_date,object
customer_id,object
city,object
category,object
product,object
quantity,object
unit_price_pkr,int64
discount_pct,object


### The grain matters
This is an **order line** dataset. One order can have several products, so a repeated `order_id` does not automatically mean an error.

In [8]:
print("Total rows:", len(df))
print("Unique orders:", df["order_id"].nunique())
print("Unique line IDs:", df["line_id"].nunique())
print("Rows per order (sample):")
print(df["order_id"].value_counts().head())

Total rows: 565
Unique orders: 280
Unique line IDs: 560
Rows per order (sample):
order_id
ORD-2025-25122    3
ORD-2025-25057    3
ORD-2025-25151    3
ORD-2025-25014    3
ORD-2025-25045    3
Name: count, dtype: int64


**My note:** `quantity` and `discount_pct` came in as text. That is a datatype problem to revisit in Level 3, not something to ignore.